# Domino's Michigan: Google Review Themes
**SI 485 capstone — team notebook.** Everyone runs this same notebook on their own slice of stores with their own free keys.

Final output: one shared **Google Sheet** with tabs `Codebook`, `Stores`, `Counties`, `Statewide`, `Tagged reviews`.

### Store ranges (158 stores)
| Person | FIRST_STORE | LAST_STORE | Area (roughly) |
|---|---|---|---|
| Julian (coordinator) | MI001 | MI032 | Adrian → Dearborn |
| Teammate 2 | MI033 | MI064 | Detroit → Grand Rapids |
| Teammate 3 | MI065 | MI096 | Grand Rapids → Lansing |
| Teammate 4 | MI097 | MI127 | Lapeer → Royal Oak |
| Teammate 5 | MI128 | MI158 | Saginaw → Ypsilanti |

### Order of work (the steps depend on each other)
1. **Everyone:** run Steps 0–2 (pulls your stores' reviews into the Sheet).
2. **Julian:** once all 5 `raw_` tabs exist, run Step 3 (builds the shared theme codebook, 1 Gemini call).
3. **Everyone:** run Steps 4–5 (tags your reviews + writes your store summaries, ~10 Gemini calls each).
4. **Julian:** run Steps 6–7 (merges everything, county + statewide summaries, charts).

### One-time setup (Colab sidebar → 🔑 **Secrets**, turn on *Notebook access* for each)
| Secret name | Where to get it | Needed by |
|---|---|---|
| `GEMINI_API_KEY` | aistudio.google.com → *Get API key* | everyone |
| `APIFY_TOKEN` | apify.com (free account, no card) → *Settings → API & Integrations* | everyone (if `REVIEW_SOURCE = "apify"`) |
| `GOOGLE_MAPS_KEY` | Google Cloud console, Places API (New) | only if `REVIEW_SOURCE = "places"` |

**Never paste a key into a code cell.** Every Gemini answer and every review pull is saved to your Google Drive (`dominos_reviews_cache` folder), so if Colab disconnects or you hit the daily limit, just rerun — finished work is never paid for twice.

## Step 0 — Your settings (edit this cell)

In [ ]:
# ===== EVERYONE: fill these in =====
MY_NAME = "julian"          # short, lowercase, no spaces. Names your tabs in the Sheet (raw_julian, ...)
FIRST_STORE = "MI001"       # your range from the table above
LAST_STORE = "MI032"
SHEET_URL = "PASTE_THE_TEAM_SHEET_URL_HERE"
I_AM_COORDINATOR = True    # only Julian sets this to True

# ===== Leave these alone unless the team agrees to change them =====
REVIEW_SOURCE = "apify"     # "apify" = newest 50 reviews per store; "places" = official API, only 5 per store
REVIEWS_PER_STORE = 50
GEMINI_MODEL = "gemini-3.8-flash"   # check AI Studio → Rate limits; use the Flash model with the most free requests/day
REVIEWS_PER_CALL = 400      # reviews sent to Gemini in one request (bigger = fewer requests used)
STORES_PER_CALL = 10        # store summaries written per request
SECONDS_BETWEEN_CALLS = 10  # pause between Gemini requests so we stay under the per-minute limit

In [ ]:
# (Colab setup) install the two libraries Colab doesn't already have
!pip install -q google-genai apify-client

In [ ]:
# Libraries used everywhere below
import os, io, json, time, re, hashlib, random, urllib.parse
import pandas as pd

class DailyLimitReached(Exception):
    """Raised when Gemini says we've used today's free requests."""

In [ ]:
# (Colab setup) connect to Google Drive, the shared Sheet, Gemini and Apify,
# and define the small helper functions that talk to them
import os, io, json, time, re, hashlib, random, urllib.parse
import pandas as pd
from google.colab import auth, drive, userdata
import gspread, requests
from google.auth import default
from google import genai
from google.genai import types


class DailyLimitReached(Exception):
    """Raised when Gemini says we've used today's free requests."""


# Google Drive: our cache folder lives here
drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/dominos_reviews_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Google Sheets: sign in once with your Google account
auth.authenticate_user()
creds, _ = default()
sheet = gspread.authorize(creds).open_by_url(SHEET_URL)

# Gemini
gemini = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))


def list_tabs():
    """Names of all tabs in the shared Sheet."""
    return [ws.title for ws in sheet.worksheets()]


def read_tab(name):
    """Read one Sheet tab into a table."""
    return pd.DataFrame(sheet.worksheet(name).get_all_records())


def write_tab(name, df):
    """Replace one Sheet tab with a table (creates the tab if needed)."""
    try:
        ws = sheet.worksheet(name)
        ws.clear()
    except gspread.WorksheetNotFound:
        ws = sheet.add_worksheet(title=name, rows=10, cols=5)
    ws.resize(rows=len(df) + 1, cols=max(len(df.columns), 1))
    clean = df.astype(object).where(df.notna(), "")
    values = [list(df.columns)] + [[v.item() if hasattr(v, "item") else v for v in row]
                                   for row in clean.values.tolist()]
    ws.update(range_name="A1", values=values, value_input_option="RAW")
    print(f"  wrote {len(df)} rows to tab '{name}'")


def ask_gemini(prompt, cache_name):
    """Send one prompt to Gemini and get JSON back.
    The answer is saved to Drive, so asking the same thing again costs nothing."""
    path = f"{CACHE_DIR}/gemini_{cache_name}.json"
    if os.path.exists(path):
        return json.load(open(path))
    for attempt in range(6):
        try:
            response = gemini.models.generate_content(
                model=GEMINI_MODEL, contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json",
                                                   temperature=0.2))
            answer = json.loads(response.text)
        except json.JSONDecodeError:
            print("  Gemini returned broken JSON, retrying...")
            continue
        except Exception as e:
            message = str(e)
            if "PerDay" in message:
                raise DailyLimitReached("Out of free Gemini requests for today. "
                                        "Rerun this cell tomorrow; finished batches are saved.")
            # 429 = too many requests this minute, 503/500 = Google's servers are busy
            if any(code in message for code in ["429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500"]):
                print("  Gemini is busy or rate-limited, waiting 60 seconds...")
                time.sleep(60)
                continue
            raise
        json.dump(answer, open(path, "w"))
        time.sleep(SECONDS_BETWEEN_CALLS)
        return answer
    raise RuntimeError("Gemini failed 6 times on " + cache_name)


def fetch_store_reviews(store):
    """Newest Google reviews for one store, as a list of dicts.
    Saved to Drive, so each store is only ever pulled once."""
    path = f"{CACHE_DIR}/reviews_{REVIEW_SOURCE}_{store.store_id}.json"
    if os.path.exists(path):
        return json.load(open(path))
    query = f"Domino's Pizza {store.street}, {store.city}, MI {store.zip}"

    if REVIEW_SOURCE == "apify":
        from apify_client import ApifyClient
        from decimal import Decimal
        apify = ApifyClient(userdata.get("APIFY_TOKEN"))

        def scrape(search_url):
            run_input = {"startUrls": [{"url": search_url}],
                         "maxReviews": REVIEWS_PER_STORE,
                         "reviewsSort": "newest",
                         "language": "en",
                         "reviewsOrigin": "google",
                         "personalData": False}          # no reviewer names
            # Safety caps: a bad search can match dozens of stores, so never pay
            # for more than ~1 store's worth of reviews per run
            run = apify.actor("compass/Google-Maps-Reviews-Scraper").call(
                run_input=run_input, max_items=REVIEWS_PER_STORE,
                max_total_charge_usd=Decimal(str(round(0.05 + REVIEWS_PER_STORE * 0.001, 2))),
                logger=None)                               # hide Apify's long logs
            # newer apify-client versions return an object, older ones a dict
            dataset_id = run["defaultDatasetId"] if isinstance(run, dict) else run.default_dataset_id
            items = list(apify.dataset(dataset_id).iterate_items())
            return [{"stars": it.get("stars"), "date": (it.get("publishedAtDate") or "")[:10],
                     "text": it.get("text") or "", "owner_reply": bool(it.get("responseFromOwnerText")),
                     "place_id": it.get("placeId"), "place_name": it.get("title") or "",
                     "place_address": it.get("address") or ""} for it in items]

        def found_this_store(reviews):
            house_number = store.street.split()[0]
            return any("domino" in r["place_name"].lower() and house_number in r["place_address"]
                       for r in reviews)

        # 1st try: search by street address
        reviews = scrape("https://www.google.com/maps/search/" + urllib.parse.quote_plus(query))
        # 2nd try: if Google didn't find that address, search for Domino's right at the store's GPS point
        if not found_this_store(reviews):
            print(f"  {store.store_id}: address search missed, retrying by map location")
            reviews += scrape(f"https://www.google.com/maps/search/Domino%27s+Pizza/@{store.lat},{store.lon},17z")
    else:
        r = requests.post("https://places.googleapis.com/v1/places:searchText",
                          headers={"X-Goog-Api-Key": userdata.get("GOOGLE_MAPS_KEY"),
                                   "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.reviews"},
                          json={"textQuery": query}).json()
        place = (r.get("places") or [{}])[0]
        reviews = [{"stars": rv.get("rating"), "date": rv.get("publishTime", "")[:10],
                    "text": rv.get("text", {}).get("text", ""), "owner_reply": False,
                    "place_id": place.get("id"), "place_name": place.get("displayName", {}).get("text", ""),
                    "place_address": place.get("formattedAddress", "")} for rv in place.get("reviews", [])]

    json.dump(reviews, open(path, "w"))
    return reviews


print("Connected. Tabs in the Sheet right now:", list_tabs())

## Step 1 — The Michigan store list (built from Domino's store directory + Census county lookup)

In [ ]:
# All 158 Michigan Domino's stores. Store IDs are fixed so everyone's work lines up.
STORES_CSV = """
store_id,street,city,zip,county,lat,lon
MI001,1113 West Beecher,Adrian,49221,Lenawee County,41.88880361341968,-84.05605852603912
MI002,5164 Lake Michigan Dr Suite A,Allendale,49401,Ottawa County,42.97214024814905,-85.91338371163943
MI003,1380 Wright Avenue,Alma,48801,Gratiot County,43.39129620358024,-84.66630008465575
MI004,650 N Ripley Blvd,Alpena,49707,Alpena County,45.059115,-83.45246
MI005,2282 South Main Street,Ann Arbor,48103,Washtenaw County,42.253568967489045,-83.75231444835663
MI006,25 Jackson Industrial Dr,Ann Arbor,48103,Washtenaw County,42.28840962309836,-83.84375189246384
MI007,2601 Plymouth Rd,Ann Arbor,48105-2487,Washtenaw County,42.3029848124437,-83.70931419163298
MI008,4910 Washtenaw Avenue,Ann Arbor,48108,Washtenaw County,42.25012703180286,-83.66264954209328
MI009,716 Packard Street,Ann Arbor,48107,Washtenaw County,42.270393,-83.74038
MI010,2795 University Drive,Auburn Hills,48326,Oakland County,42.67016575392891,-83.2335737164612
MI011,2545 Capital Ave SW,Battle Creek,49015,Calhoun County,42.2652264,-85.1978063
MI012,780 Capital Ave NE,Battle Creek,49017,Calhoun County,42.32980187179665,-85.15511536441801
MI013,4041 N. Euclid Road,Bay City,48706,Bay County,43.62175310449707,-83.91355276107788
MI014,521 Washington,Bay City,48708,Bay County,43.58022479949578,-83.88493316797735
MI015,9588 Belleville Rd,Belleville,48111,Wayne County,42.233164814722784,-83.48660096642789
MI016,1927 Pipestone Rd,Benton Harbor,49022,Berrien County,42.08299299999999,-86.41799499999999
MI017,840 South State Street,Big Rapids,49307,Mecosta County,43.688678362910366,-85.48468897056966
MI018,222 Grand River Ave,Brighton,48116,Livingston County,42.5309288,-83.7805457
MI019,1435 N. Mitchell St,Cadillac,49061,Wexford County,44.27148596479349,-85.40697321295738
MI020,39731 Ford Rd,Canton,48187,Wayne County,42.3228756,-83.4322594
MI021,44350 Cherry Hill Road,Canton,48187,Wayne County,42.30846051063144,-83.47625528624121
MI022,328 Lansing St,Charlotte,48813,Eaton County,42.568695842916505,-84.8262999238629
MI023,7081 Dixie Hwy,Clarkston,48346,Oakland County,42.72298752744232,-83.42619613535389
MI024,1226 W 14 Mile Rd,Clawson,48017,Oakland County,42.53365761797501,-83.16438778434771
MI025,35785 Harper Rd,Clinton Township,48035,Macomb County,42.56035732391627,-82.8708485663264
MI026,39880 Garfield Road,Clinton Township,48038,Macomb County,42.590164865810586,-82.9513109309283
MI027,3520 W Vienna Rd,Clio,48420,Genesee County,43.17784752853282,-83.75368835581969
MI028,40230 W 14 Mile Rd,Commerce Twp,48390,Oakland County,42.52802531192888,-83.44349501493717
MI029,740 Benstein Road,Commerce,48390,Oakland County,42.533374517303365,-83.50136965513225
MI030,4038 Alpine Ave NW,comstock park,49321,Kent County,43.036734751699605,-85.68868421730139
MI031,825 South State Rd,Davison,48423,Genesee County,43.02098091948271,-83.51656693189094
MI032,23960 Carlysle Street,Dearborn,48124,Wayne County,42.28396706290059,-83.27045234933354
MI033,10437 West McNichols,Detroit,48221,Wayne County,42.4165698702886,-83.16547393798828
MI034,13250 Gratiot Ave,Detroit,48205,Wayne County,42.42366342247314,-82.98480972647667
MI035,15803 Mack Ave,Detroit,48224,Wayne County,42.38965532536789,-82.93711149615558
MI036,16005 W McNichols Road,Detroit,48235,Wayne County,42.415586,-83.2048394
MI037,1636 E Eight Mile Rd,Detroit,48203,Wayne County,42.44655254809154,-83.08753658832757
MI038,2642 East Jefferson Avenue,Detroit,48207,Wayne County,42.337702358953315,-83.02097623304219
MI039,4501 Woodward Ave,Detroit,48201,Wayne County,42.353589,-83.062485
MI040,6171 Vernor Hwy Suite B,Detroit,48209,Wayne County,42.3155463,-83.1077804
MI041,143 N Harrison Rd,East Lansing,48823,Ingham County,42.734863687762356,-84.49386558454235
MI042,3992 Coleman Rd,East Lansing,48823,Clinton County,42.77068808331917,-84.48406101278357
MI043,15143 E 10 Mile Rd.,Eastpointe,48021,Macomb County,42.479913,-82.965016
MI044,4058 W Jefferson Ave,Ecorse,48229,Wayne County,42.25114885745722,-83.14136610113279
MI045,2313 Ludington St,Escanaba,49829,Delta County,45.74538294019055,-87.08099150481742
MI046,30336 W 9 Mile Road,Farmington Hills,48336,Oakland County,42.45631904645391,-83.34624409675598
MI047,34781 Grand River Dr,Farmington,48335,Oakland County,42.46828984734199,-83.3906662088838
MI048,913 N Leroy St,Fenton,48430,Genesee County,42.80702300674139,-83.70714328722443
MI049,26662 Telegraph Rd,Flat Rock,48134,Wayne County,42.112042218786044,-83.28411152558489
MI050,2113 S Ballenger Hwy,Flint,48503,Genesee County,42.99295942829058,-83.73247414827341
MI051,3718 E. Court Street,Flint,48506,Genesee County,43.018295011739504,-83.63673001527786
MI052,4402 Fenton Road,Flint,48507,Genesee County,42.9837162733622,-83.6931811273098
MI053,5040 W Pierson Rd,Flint,48504,Genesee County,43.061138341555,-83.77440740169732
MI054,4019 24th Ave.,Fort Gratiot,48059,St. Clair County,43.0256844825257,-82.45650172233582
MI055,16042 15 Mile Road,Fraser,48026,Macomb County,42.55280135215099,-82.95964041355546
MI056,30068 Ford Rd,Garden City,48135,Wayne County,42.32592177819939,-83.33772730059434
MI057,1060 Main Street,Gaylord,49735,Otsego County,45.02787030954687,-84.69310389206989
MI058,21 Lowrie Drive,Gladstone,49837,Delta County,45.83702961257402,-87.03720778226852
MI059,130 Perry Rd,Grand Blanc,48439,Genesee County,42.9239517,-83.6245749
MI060,311 N 7th Street,Grand Haven,49417,Ottawa County,43.06453721161835,-86.22165084380984
MI061,327 S Bridge St,Grand Ledge,48837,Eaton County,42.75256169525364,-84.74695602943046
MI062,1025 Division Ave S,Grand Rapids,49507,Kent County,42.94439169322687,-85.6678019464016
MI063,1335 Lake Dr SE,Grand Rapids,49506,Kent County,42.957926176904095,-85.63559800386429
MI064,3928 Plainfield Ave NE,Grand Rapids,49525,Kent County,43.034123285781476,-85.62032818794256
MI065,6159 Kalamazoo Ave SE,Grand Rapids,49508,Kent County,42.85156164593546,-85.62646937725187
MI066,619 Leonard St NW,Grand Rapids,49504,Kent County,42.98581355742178,-85.68353437678091
MI067,6539 28th St SE,Grand Rapids,49546,Kent County,42.9136658,-85.5065231
MI068,900 Fulton St West,Grand Rapids,49504,Kent County,42.96340098920885,-85.69033926595591
MI069,1720 W Washington St,Greenville,48838,Montcalm County,43.17775035663493,-85.28123038485022
MI070,20431 Mack Ave,Grosse Pointe,48236,Wayne County,42.439501918637134,-82.9080790980106
MI071,3535 Cannif St,Hamtramck,48212,Wayne County,42.402508170918686,-83.05341066595594
MI072,1180 W M43 Highway B,Hastings,49058,Barry County,42.648549,-85.312405
MI073,14331 Woodward Ave,Highland Park,48203,Wayne County,42.40809713034838,-83.0997120004932
MI074,264 N River Ave # 20,Holland,49424,Ottawa County,42.805727799953765,-86.11564996267589
MI075,738 Michigan Ave,Holland,49423,Ottawa County,42.76992023986538,-86.11644194419765
MI076,4048 Grange Hall Road,Holly,48442,Oakland County,42.806056354805285,-83.62489879131311
MI077,2068 Cedar Street,Holt,48842,Ingham County,42.64256463125218,-84.51726973056793
MI078,200 Pearl St.,Houghton,49931,Houghton County,47.12004108020293,-88.5547151369883
MI079,1900 N Latson Rd,Howell,48843,Livingston County,42.62848102365558,-83.87470820549424
MI080,120 S Dexter St,Ionia,48846,Ionia County,42.9816075,-85.0696336
MI081,1200 S Stephenson Ave,Iron Mountain,49801,Dickinson County,45.81108704146585,-88.05953832136424
MI082,135 E. Aurora St.,Ironwood,49938,Gogebic County,46.45431910706586,-90.1683692022152
MI083,1236 Francis St,Jackson,49203,Jackson County,42.233070856906764,-84.40420730061572
MI084,3246 E Michigan Ave,Jackson,49202,Jackson County,42.25706901323869,-84.3619601586215
MI085,314 Baldwin St,Jenison,49428,Ottawa County,42.90671586013421,-85.78926309943193
MI086,701 Olds St,Jonesville,49250,Hillsdale County,41.96526899351742,-84.66834258371021
MI087,2706 W. Michigan Ave.,Kalamazoo,49006,Kalamazoo County,42.281588332432904,-85.62276162207127
MI088,4129 Portage Street,Kalamazoo,49001,Kalamazoo County,42.24937883982056,-85.55972509877662
MI089,5585 Gull Road #108,Kalamazoo,46240,Kalamazoo County,42.3254037,-85.5199893
MI090,3151 Breton Rd SE,Kentwood,49512,Kent County,42.90583397350874,-85.60839504003525
MI091,1019 E. State Road,Lansing,48906,Clinton County,42.78468701382508,-84.54189509153366
MI092,1701 S. Waverly Road,Lansing,48917,Ingham County,42.715337,-84.602129
MI093,2015 N Larch St,Lansing,48906,Ingham County,42.756243220167605,-84.54474491739717
MI094,234 S Washington Square,Lansing,48933,Ingham County,42.731438,-84.552626
MI095,4800 W Saginaw Hwy,Lansing,48917,Eaton County,42.74144817362527,-84.61712260545937
MI096,5645 S Cedar St,Lansing,48911,Ingham County,42.67548071239667,-84.54669314055991
MI097,858 S Main St,Lapeer,48446,Lapeer County,43.03948313872016,-83.32192810109223
MI098,28651 Southfield Rd,Lathrup Village,48076,Oakland County,42.500213748039165,-83.2228410243988
MI099,1318 Dix Highway,Lincoln Park,48146,Wayne County,42.26325453844395,-83.18684577941895
MI100,16136 Middlebelt Road,Livonia,48154,Wayne County,42.4057869,-83.3337239
MI101,27609 Plymouth Road,Livonia,48150,Wayne County,42.369267,-83.315473
MI102,37298 Six Mile Road,Livonia,48152,Wayne County,42.411479310120214,-83.41243990549272
MI103,5784 US 10,Ludington,49431,Mason County,43.956576,-86.423833
MI104,21697 21 Mile Rd.,Macomb Twp.,48044,Macomb County,42.644994258307854,-82.89760424765518
MI105,206 Cypress,Manistee,49660,Manistee County,44.24470102552826,-86.3191663738185
MI106,641 W Washington Street,Marquette,49855,Marquette County,46.54486202063232,-87.40579962730408
MI107,912 W Michigan Ave,Marshall,49068,Calhoun County,42.272536,-84.975293
MI108,1311 10Th St,Menominee,49858,Menominee County,45.11116,-87.61471
MI109,2804 Rodd St,Midland,48640,Midland County,43.62315213883765,-84.22648798038608
MI110,1929 North Telegraph Road,Monroe,48162,Monroe County,41.94962093705455,-83.40070600426493
MI111,706 S Mission St,Mount Pleasant,48858,Isabella County,43.59779016156218,-84.76799552144257
MI112,1332 Apple Ave,Muskegon,49442,Muskegon County,43.23447831559474,-86.21108002494111
MI113,3275 Henry Street,Muskegon,49441,Muskegon County,43.193031604609004,-86.26402214169497
MI114,6022 Harvey St,Muskegon,49444,Muskegon County,43.1431024794213,-86.20535914601533
MI115,35346 23 Mile Rd.,New Baltimore,48047,Macomb County,42.67568933968687,-82.7499376684591
MI116,41728 W 10 Mile Road,Novi,48375,Oakland County,42.46879089999999,-83.4568733
MI117,13735 W 9 Mile Rd.,Oak Park,48237,Oakland County,42.4594081262653,-83.18396868730821
MI118,4750 Central Park Dr,Okemos,48864,Ingham County,42.718638042961324,-84.40872073173523
MI119,616 W Main St,Owosso,48867,Shiawassee County,42.997836439255174,-84.17941331863403
MI120,979 S Lapeer Rd,Oxford,48371,Oakland County,42.811430321947235,-83.25169812253193
MI121,412 Oaks Crossing,Plainwell,49080,Allegan County,42.44786944273764,-85.6725961116356
MI122,1043 Ann Arbor Road,Plymouth,48170,Wayne County,42.3582808,-83.4729667
MI123,1420 24Th St,Port Huron,48060,St. Clair County,42.9681537740938,-82.45195521307826
MI124,1788 W Milham Avenue,Portage,49024,Kalamazoo County,42.2305006403074,-85.60998921097962
MI125,121 East University,Rochester,48307,Oakland County,42.682323081225476,-83.1328641932214
MI126,400 E Division St NE,Rockford,49341,Kent County,43.116703138261826,-85.55004908465577
MI127,1317 East Eleven Mile,Royal Oak,48067,Oakland County,42.490442230572604,-83.1286602169439
MI128,5560 State St,Saginaw,48603,Saginaw County,43.437288,-84.024288
MI129,27883 Harper Rd.,Saint Clair Shores,48081,Macomb County,42.49751147454144,-82.89803679396749
MI130,401 E Michigan Ave,Saline,48176,Washtenaw County,42.170333910463945,-83.77399434150902
MI131,719 Ashmun St.,Sault Ste Maire,49783,Chippewa County,46.4939959,-84.3498912
MI132,55025 Van Dyke,Shelby Township,48316,Macomb County,42.69899038873311,-83.03529211882051
MI133,22235 Pontiac Trail,South Lyon,48178,Oakland County,42.443605,-83.65191
MI134,28803 Northwestern Highway,Southfield,48034,Oakland County,42.499021,-83.306149
MI135,14100 Fort Street,Southgate,48195,Wayne County,42.2056583,-83.1809433
MI136,17675 Eureka Road,Southgate,48195,Wayne County,42.19827379290383,-83.22175741195679
MI137,630 S State St,Sparta,49345,Kent County,43.145326895312685,-85.70932656526566
MI138,101 N Clinton Ave,St Johns,48879,Clinton County,43.001593003981434,-84.55880105495447
MI139,2080 Niles Road #100,St Joseph,49085,Berrien County,42.07871156756812,-86.48409605026245
MI140,2767 Fourteen Mile Road,Sterling Heights,48310,Macomb County,42.53577867377527,-83.0786755934992
MI141,5930 Cleveland Avenue,Stevensville,49127,Berrien County,42.01041622122091,-86.49549007415771
MI142,1139 S Centerville Rd,Sturgis,49091,St. Joseph County,41.7871284,-85.429657
MI143,21751 Ecorse Road,Taylor,48180,Wayne County,42.255805,-83.248717
MI144,3480 W South Airport Rd,Traverse City,49684,Grand Traverse County,44.72406982225364,-85.64480010648685
MI145,748 Munson Ave,Traverse City,49686,Grand Traverse County,44.754531,-85.572555
MI146,3327 West Rd,Trenton,48183,Wayne County,42.14037555399014,-83.20956391544085
MI147,3843 Rochester Rd,Troy,48083,Oakland County,42.57498192893549,-83.12912692131249
MI148,8771 Hall Road,Utica,48317,Macomb County,42.626321605147886,-83.02418327657506
MI149,13734 12 Mile Rd,Warren,48088,Macomb County,42.5065384,-82.9864502
MI150,6115 East 10 Mile Rd,Warren,48091,Macomb County,42.47763919106911,-83.04270252585411
MI151,3925 Elizabeth Lake Rd.,Waterford,48328,Oakland County,42.6387329,-83.36493
MI152,35171 E Michigan Ave,Wayne,48184,Wayne County,42.27865221053596,-83.3854485955726
MI153,8017 N Wayne Road,Westland,48185,Wayne County,42.346258355990095,-83.38610532311918
MI154,10531 Highland Road,White Lake,48386,Oakland County,42.6501892,-83.4941904
MI155,9567 Main St,Whitmore Lake,48189,Washtenaw County,42.42331428062966,-83.76226017790987
MI156,3596 Clyde Park Ave SW,Wyoming,49509,Kent County,42.89917686397768,-85.68453721236432
MI157,2121 S Grove Rd,Ypsilanti,48198,Washtenaw County,42.21685317215299,-83.5675485
MI158,953 Washtenaw Ave,Ypsilanti,48197,Washtenaw County,42.24581818212228,-83.62764649094521
"""
stores = pd.read_csv(io.StringIO(STORES_CSV.strip()), dtype={"zip": str})
my_stores = stores[(stores.store_id >= FIRST_STORE) & (stores.store_id <= LAST_STORE)]
print(f"{len(stores)} stores in Michigan; you have {len(my_stores)} ({FIRST_STORE}–{LAST_STORE})")
my_stores.head()

In [ ]:
# Counting helpers used in Steps 5-7 (plain Python, no Gemini)
codes = {}   # theme code -> theme name, filled in once the Codebook exists

def theme_shares(df, top=5):
    """% of reviews mentioning each theme, biggest first."""
    counts = df.theme_codes.str.split(";").explode().loc[lambda s: s != ""].value_counts()
    return [(codes[c], round(100 * n / len(df))) for c, n in counts.head(top).items()]

def describe(df):
    stars = pd.to_numeric(df.stars, errors="coerce")
    return {"n_reviews": len(df), "avg_stars": round(stars.mean(), 2),
            "pct_negative": round(100 * (stars <= 2).mean())}

## Step 2 — Pull the newest reviews for your stores (everyone)
Takes ~1 minute per store with Apify. Uses well under $1 of Apify's free $5/month. **No Gemini requests.**

In [ ]:
def pick_this_store(reviews, store):
    """A Google Maps search can return more than one place. Keep only the Domino's
    whose address has this store's house number (or the first Domino's found)."""
    dominos = [r for r in reviews if "domino" in r["place_name"].lower()]
    house_number = store.street.split()[0]
    matching = [r for r in dominos if house_number in r["place_address"]]
    chosen = matching or dominos
    if not chosen:
        return []
    first_place = chosen[0]["place_id"]
    return [r for r in chosen if r["place_id"] == first_place]

raw_rows = []
for store in my_stores.itertuples():
    reviews = pick_this_store(fetch_store_reviews(store), store)
    # drop star-only reviews (no text = nothing to find themes in) and exact duplicates
    seen = set()
    kept = []
    for r in sorted(reviews, key=lambda r: r["date"], reverse=True):
        text = " ".join(r["text"].split())
        if text and text not in seen:
            seen.add(text)
            kept.append({**r, "text": text})
    for n, r in enumerate(kept, 1):
        raw_rows.append({"review_id": f"{store.store_id}-{n:03d}", "store_id": store.store_id,
                         "stars": r["stars"], "date": r["date"], "owner_replied": r["owner_reply"],
                         "text": r["text"], "google_address": r["place_address"]})
    note = ""
    if not kept:
        note = "   <-- CHECK: no reviews found"
    elif store.street.split()[0] not in kept[0]["place_address"]:
        note = f"   <-- CHECK: Google matched a different address: {kept[0]['place_address']}"
        note = ""
    if not kept:
        note = "   <-- CHECK: no reviews found"
    elif store.street.split()[0] not in kept[0]["place_address"]:
        note = f"   <-- CHECK: Google matched a different address: {kept[0]['place_address']}"
    print(f"{store.store_id} {store.city:<18} {len(kept):>3} reviews with text{note}")
raw = pd.DataFrame(raw_rows)
print(f"\n{len(raw)} reviews total")
write_tab(f"raw_{MY_NAME}", raw)

In [ ]:
# FULL SCRAPE, part 1: how many Google reviews does each of my stores have?
# One small Apify run (1 review per store), costs about 2 cents
from apify_client import ApifyClient
from decimal import Decimal

apify = ApifyClient(userdata.get("APIFY_TOKEN"))
PRICE_PER_REVIEW = 0.0005   # Apify's listed price: $0.50 per 1,000 reviews


def place_id_for(store):
    """The Google place ID of this store, taken from the reviews we already pulled
    (only if they came from the right address)."""
    path = f"{CACHE_DIR}/reviews_apify_{store.store_id}.json"
    if not os.path.exists(path):
        return None
    house_number = store.street.split()[0]
    ids = [r["place_id"] for r in json.load(open(path))
           if "domino" in r["place_name"].lower() and house_number in r["place_address"]]
    return ids[0] if ids else None


def run_reviews(place_ids, max_reviews, max_charge_usd):
    """One Apify run for the given Google place IDs. Returns the raw review items."""
    run = apify.actor("compass/Google-Maps-Reviews-Scraper").call(
        run_input={"placeIds": place_ids, "maxReviews": max_reviews, "reviewsSort": "newest",
                   "language": "en", "reviewsOrigin": "google", "personalData": False},
        max_items=max_reviews * len(place_ids),
        max_total_charge_usd=Decimal(str(round(max_charge_usd, 2))),
        logger=None)
    return list(apify.dataset(run.default_dataset_id).iterate_items())


place_ids = {}
for store in my_stores.itertuples():
    pid = place_id_for(store)
    if pid:
        place_ids[store.store_id] = pid
    else:
        print(f"{store.store_id} skipped (no confirmed Google listing yet)")

count_path = f"{CACHE_DIR}/review_counts.json"
if os.path.exists(count_path):
    total_by_place = json.load(open(count_path))
else:
    items = run_reviews(list(place_ids.values()), 1, 0.10)
    total_by_place = {it.get("placeId"): it.get("reviewsCount") or 0 for it in items}
    json.dump(total_by_place, open(count_path, "w"))

counts = pd.DataFrame([{"store_id": sid, "place_id": pid, "total_reviews": total_by_place.get(pid, 0)}
                       for sid, pid in place_ids.items()])
counts = counts.merge(stores[["store_id", "city"]], on="store_id")
counts["est_cost_usd"] = (counts.total_reviews * PRICE_PER_REVIEW).round(2)
print(f"{len(counts)} stores, {counts.total_reviews.sum():,} Google reviews in total")
print(f"Estimated cost to scrape them ALL: ${counts.est_cost_usd.sum():.2f}")
counts.sort_values("total_reviews")

In [ ]:
# FULL SCRAPE, part 2: every review for as many stores as the budget allows.
# Smallest stores go first, so the budget finishes the most stores possible.
BUDGET_USD = 5.0   # check apify.com -> Billing/Usage and set this to what's left of your $5

spent = 0
full_rows = []
for c in counts.sort_values("total_reviews").itertuples():
    path = f"{CACHE_DIR}/full_{c.store_id}.json"
    if os.path.exists(path):
        items = json.load(open(path))                 # already scraped: free now, but it counts toward the budget
    else:
        if spent + c.est_cost_usd > BUDGET_USD:
            print(f"{c.store_id} {c.city:<18} skipped: {c.total_reviews} reviews (~${c.est_cost_usd}) would go over budget")
            continue
        items = run_reviews([c.place_id], c.total_reviews + 25, c.est_cost_usd * 1.3 + 0.05)
        items = [{"stars": it.get("stars"), "date": (it.get("publishedAtDate") or "")[:10],
                  "text": " ".join((it.get("text") or "").split()),
                  "owner_replied": bool(it.get("responseFromOwnerText"))} for it in items]
        json.dump(items, open(path, "w"))
        spent += len(items) * PRICE_PER_REVIEW
    print(f"{c.store_id} {c.city:<18} {len(items):>5} of {c.total_reviews} reviews   (spent so far ~${spent:.2f})")
    for n, it in enumerate(sorted(items, key=lambda r: r["date"], reverse=True), 1):
        full_rows.append({"review_id": f"{c.store_id}-F{n:04d}", "store_id": c.store_id, **it})

full = pd.DataFrame(full_rows)
full["has_text"] = full.text.str.len() > 0
print(f"\n{len(full):,} reviews total, {full.has_text.sum():,} with text")
full.to_csv(f"{CACHE_DIR}/full_reviews_{MY_NAME}.csv", index=False)   # backup copy in Drive
write_tab(f"full_{MY_NAME}", full)

In [ ]:
# Retry stores where Google stopped early (it's random; a retry usually works)
from decimal import Decimal

SORTS = ["mostRelevant", "newest", "lowestRanking"]   # order doesn't matter when we take every review

for c in counts.itertuples():
    path = f"{CACHE_DIR}/full_{c.store_id}.json"
    have = len(json.load(open(path))) if os.path.exists(path) else 0
    if have >= 0.9 * c.total_reviews:
        continue                                        # already (nearly) complete
    for attempt, sort in enumerate(SORTS, 1):
        run = apify.actor("compass/Google-Maps-Reviews-Scraper").call(
            run_input={"placeIds": [c.place_id], "maxReviews": c.total_reviews + 25,
                       "reviewsSort": sort, "language": "en",
                       "reviewsOrigin": "google", "personalData": False},
            max_items=c.total_reviews + 25,
            max_total_charge_usd=Decimal(str(round(c.est_cost_usd * 1.3 + 0.05, 2))),
            logger=None)
        items = list(apify.dataset(run.default_dataset_id).iterate_items())
        print(f"{c.store_id} {c.city:<18} try {attempt} ({sort}): {len(items)} of {c.total_reviews}")
        if len(items) > have:                           # keep the best attempt
            have = len(items)
            json.dump([{"stars": it.get("stars"), "date": (it.get("publishedAtDate") or "")[:10],
                        "text": " ".join((it.get("text") or "").split()),
                        "owner_replied": bool(it.get("responseFromOwnerText"))} for it in items],
                      open(path, "w"))
        if have >= 0.9 * c.total_reviews:
            break
        time.sleep(60)                                  # give Google a minute before trying again

## Step 3 — Build the theme codebook (Julian only, after all 5 `raw_` tabs exist)
Gemini reads a sample of ~300 reviews (about 2 per store) from across the state and proposes the themes. **1 Gemini request.**
After it runs, read the `Codebook` tab together as a team. You can edit theme names/definitions right in the Sheet before anyone runs Step 4.

In [ ]:
if not I_AM_COORDINATOR:
    print("Skip — Julian runs this step.")
else:
    raw_tabs = [f"full_{MY_NAME}"]
    print("Using tabs:", raw_tabs)
    everyone = pd.concat([read_tab(t) for t in raw_tabs], ignore_index=True)
    everyone = everyone[everyone.text.astype(str).str.strip() != ""]   # written reviews only

    # Sample ~2 reviews from every store (so every area is represented), topped up at random to at least 300
    per_store = everyone.groupby("store_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 2), random_state=485))
    rest = everyone.drop(per_store.index)
    top_up = max(0, min(len(rest), 300 - len(per_store)))
    sample = pd.concat([per_store, rest.sample(top_up, random_state=485)])

    lines = "\n".join(f"{r.stars}★ | {str(r.text)[:500]}" for r in sample.itertuples())
    prompt = f"""You are helping a university consulting team analyze Google reviews of Domino's Pizza stores in Michigan.
Below are {len(sample)} customer reviews (star rating | text).

Create a codebook of 10 to 14 themes that together cover what customers talk about.
Rules:
- Make themes directional when it matters (e.g. "Late or slow delivery" and "Fast delivery" are separate themes).
- Themes must be specific enough for a store manager to act on (not just "good experience").
- The last theme must be "Other" for reviews that fit nothing else.
Return JSON: a list of objects with keys "code" (T01, T02, ...), "theme" (2-5 words),
"definition" (one sentence), "example" (a short quote from the reviews, under 15 words).

REVIEWS:
{lines}"""
    codebook = pd.DataFrame(ask_gemini(prompt, "codebook"))
    write_tab("Codebook", codebook)
    display(codebook)

## Step 4 — Tag every one of your reviews with themes (everyone, after the Codebook exists)
~250 reviews per request → about **7 Gemini requests** for 32 stores.

In [ ]:
codebook = read_tab("Codebook")
codes = dict(zip(codebook.code, codebook.theme))
codebook_text = "\n".join(f"{r.code} = {r.theme}: {r.definition}" for r in codebook.itertuples())

raw = read_tab(f"full_{MY_NAME}")
raw = raw[raw.text.astype(str).str.strip() != ""].reset_index(drop=True)   # written reviews only
review_ids = list(raw.review_id)
lookup = raw.set_index("review_id")

def tag_batch(ids, label):
    lines = "\n".join(f"{i} | {lookup.at[i, 'stars']}★ | {str(lookup.at[i, 'text'])[:600]}" for i in ids)
    prompt = f"""Tag each Domino's Pizza review with themes from this codebook.

CODEBOOK:
{codebook_text}

Rules: give each review 1 to 3 theme codes, most important first. Use the "Other" code only if nothing fits.
Return JSON: one object mapping every review ID to its list of codes, e.g. {{"MI001-004": ["T01", "T05"]}}.
Include every review ID exactly once.

REVIEWS (ID | stars | text):
{lines}"""
    batch_key = hashlib.md5("".join(ids).encode()).hexdigest()[:8]
    return ask_gemini(prompt, f"tag_{MY_NAME}_{label}_{batch_key}")

tags = {}
batches = [review_ids[i:i + REVIEWS_PER_CALL] for i in range(0, len(review_ids), REVIEWS_PER_CALL)]
for n, ids in enumerate(batches, 1):
    print(f"batch {n}/{len(batches)} ({len(ids)} reviews)")
    tags.update(tag_batch(ids, f"b{n:02d}"))

def valid_codes(review_id):
    return [c for c in tags.get(review_id, []) if c in codes][:3]

# Gemini sometimes skips a few reviews; send the skipped ones once more
missing = [i for i in review_ids if not valid_codes(i)]
if missing:
    print(f"{len(missing)} reviews came back untagged; retrying them in one request")
    tags.update(tag_batch(missing, "retry"))
    print(f"still untagged: {sum(not valid_codes(i) for i in review_ids)}")

raw["theme_codes"] = [";".join(valid_codes(i)) for i in raw.review_id]
raw["themes"] = [";".join(codes[c] for c in valid_codes(i)) for i in raw.review_id]
write_tab(f"tagged_{MY_NAME}", raw)
raw.head()

## Step 5 — Write a short summary for each of your stores (everyone)
Python does the counting (free); Gemini only writes the sentences. 10 stores per request → about **3 Gemini requests**.

In [ ]:
def store_block(store, df):
    s = describe(df)
    themes = ", ".join(f"{t} {p}%" for t, p in theme_shares(df))
    low = df[pd.to_numeric(df.stars, errors="coerce") <= 2].text.head(2)
    high = df[pd.to_numeric(df.stars, errors="coerce") >= 4].text.head(1)
    quotes = "; ".join(f'"{str(q)[:200]}"' for q in list(low) + list(high))
    return (f"[{store.store_id}] {store.city} ({store.county}) — {s['n_reviews']} reviews, "
            f"avg {s['avg_stars']} stars, {s['pct_negative']}% negative (1-2 stars)\n"
            f"Top themes: {themes}\nSample reviews: {quotes}")

tagged = read_tab(f"tagged_{MY_NAME}")
tagged["theme_codes"] = tagged.theme_codes.astype(str).replace("nan", "")
have_reviews = [s for s in my_stores.itertuples() if (tagged.store_id == s.store_id).any()]

summaries = {}
groups = [have_reviews[i:i + STORES_PER_CALL] for i in range(0, len(have_reviews), STORES_PER_CALL)]
for n, group in enumerate(groups, 1):
    blocks = "\n\n".join(store_block(s, tagged[tagged.store_id == s.store_id]) for s in group)
    prompt = f"""You are writing part of a consulting report for Domino's Pizza.
For each store below, write 2-3 plain sentences: what customers praise, what they complain about,
and the one thing the store should fix first. Use only the data given; do not invent numbers.
Return JSON mapping each store ID to its summary, e.g. {{"MI001": "..."}}.

STORES:
{blocks}"""
    print(f"group {n}/{len(groups)}")
    summaries.update(ask_gemini(prompt, f"stores_{MY_NAME}_g{n:02d}_" + hashlib.md5(blocks.encode()).hexdigest()[:8]))

rows = []
for s in my_stores.itertuples():
    df = tagged[tagged.store_id == s.store_id]
    top = theme_shares(df, 3) if len(df) else []
    rows.append({"store_id": s.store_id, "street": s.street, "city": s.city, "county": s.county,
                 **(describe(df) if len(df) else {"n_reviews": 0, "avg_stars": "", "pct_negative": ""}),
                 **{f"top_theme_{k+1}": f"{t} ({p}%)" for k, (t, p) in enumerate(top)},
                 "summary": summaries.get(s.store_id, "")})
store_table = pd.DataFrame(rows)
write_tab(f"stores_{MY_NAME}", store_table)
store_table.head()

In [ ]:
# Top 2 COMPLAINT themes per store (plain Python, no Gemini)
# These are the negative themes from the Codebook; "T14" only counts if you added that row
NEGATIVE = ["T02", "T04", "T06", "T07", "T09", "T10", "T11", "T12", "T14"]

def top_complaints(df, top=2):
    """% of a store's reviews mentioning each complaint theme, biggest first."""
    counts = df.theme_codes.str.split(";").explode()
    counts = counts[counts.isin(NEGATIVE)].value_counts().head(top)
    return [f"{codes[c]} ({round(100 * n / len(df))}%)" for c, n in counts.items()]

for k in [1, 2]:
    store_table[f"top_complaint_{k}"] = ""
for i, s in store_table.iterrows():
    df = tagged[tagged.store_id == s.store_id]
    for k, text in enumerate(top_complaints(df) if len(df) else [], 1):
        store_table.at[i, f"top_complaint_{k}"] = text

write_tab(f"stores_{MY_NAME}", store_table)
store_table[["store_id", "city", "pct_negative", "top_complaint_1", "top_complaint_2"]]

## Step 6 — Merge everyone's work + county and statewide summaries (Julian only)
Run after all 5 teammates finish Step 5. About **6 Gemini requests**.

In [ ]:
if not I_AM_COORDINATOR:
    print("Skip — Julian runs this step.")
else:
    codebook = read_tab("Codebook")
    codes = dict(zip(codebook.code, codebook.theme))
    tabs = list_tabs()
    print("Merging:", [t for t in tabs if t.startswith(("tagged_", "stores_"))])
    all_tagged = pd.concat([read_tab(t) for t in tabs if t.startswith("tagged_")], ignore_index=True)
    all_tagged["theme_codes"] = all_tagged.theme_codes.astype(str).replace("nan", "")
    all_tagged = all_tagged.merge(stores[["store_id", "city", "county"]], on="store_id")
    all_stores = pd.concat([read_tab(t) for t in tabs if t.startswith("stores_")], ignore_index=True)
    all_stores = all_stores.sort_values("store_id")

    # --- County summaries: Python counts, Gemini writes (10 counties per request) ---
    county_names = sorted(all_tagged.county.unique())
    def county_block(c):
        df = all_tagged[all_tagged.county == c]
        s = describe(df)
        themes = ", ".join(f"{t} {p}%" for t, p in theme_shares(df))
        return (f"[{c}] {df.store_id.nunique()} stores, {s['n_reviews']} reviews, avg {s['avg_stars']} stars, "
                f"{s['pct_negative']}% negative\nTop themes: {themes}")
    county_summaries = {}
    for n in range(0, len(county_names), 10):
        blocks = "\n\n".join(county_block(c) for c in county_names[n:n + 10])
        prompt = f"""You are writing part of a consulting report for Domino's Pizza about Michigan counties.
For each county below, write 2 plain sentences on what stands out in customer reviews there.
Use only the data given; do not invent numbers.
Return JSON mapping each county name (exactly as written in brackets) to its summary.

COUNTIES:
{blocks}"""
        county_summaries.update(ask_gemini(prompt, f"counties_{n:03d}_" + hashlib.md5(blocks.encode()).hexdigest()[:8]))

    county_rows = []
    for c in county_names:
        df = all_tagged[all_tagged.county == c]
        top = theme_shares(df, 3)
        county_rows.append({"county": c, "stores": df.store_id.nunique(), **describe(df),
                            **{f"top_theme_{k+1}": f"{t} ({p}%)" for k, (t, p) in enumerate(top)},
                            "summary": county_summaries.get(c, "")})
    county_table = pd.DataFrame(county_rows).sort_values("n_reviews", ascending=False)

    # --- Statewide summary: 1 request ---
    s = describe(all_tagged)
    state_themes = "\n".join(f"- {t}: {p}% of reviews" for t, p in theme_shares(all_tagged, 14))
    worst = county_table[county_table.n_reviews >= 50].sort_values("pct_negative", ascending=False)
    county_lines = "\n".join(f"- {r.county}: {r.pct_negative}% negative, top theme {getattr(r, 'top_theme_1', 'n/a')}"
                             for r in worst.itertuples())
    prompt = f"""You are writing the statewide section of a consulting report for Domino's Pizza.
Michigan: {all_tagged.store_id.nunique()} stores, {s['n_reviews']} recent Google reviews, avg {s['avg_stars']} stars, {s['pct_negative']}% negative.
Theme frequency:
{state_themes}
Counties with 50+ reviews, most negative first:
{county_lines}

Use only the data given; do not invent numbers.
Return JSON with keys "summary" (one paragraph), "key_findings" (list of 5 short bullets),
"recommendations" (list of 3 short, specific actions for Domino's)."""
    state = ask_gemini(prompt, "statewide_" + hashlib.md5(prompt.encode()).hexdigest()[:8])
    statewide = pd.DataFrame(
        [{"section": "Summary", "text": state["summary"]}]
        + [{"section": "Key finding", "text": t} for t in state["key_findings"]]
        + [{"section": "Recommendation", "text": t} for t in state["recommendations"]]
        + [{"section": "Theme frequency", "text": f"{t}: {p}%"} for t, p in theme_shares(all_tagged, 14)])

    # --- Write the final tabs ---
    final_reviews = all_tagged[["review_id", "store_id", "city", "county", "stars", "date",
                                "themes", "owner_replied", "text"]].sort_values("review_id")
    write_tab("Stores", all_stores)
    write_tab("Counties", county_table)
    write_tab("Statewide", statewide)
    write_tab("Tagged reviews", final_reviews)
    display(statewide)

## Step 7 — Charts (Julian, after Step 6)

In [ ]:
import plotly.express as px

# Which codebook themes are praise vs. complaints
POSITIVE = ["T01", "T03", "T05", "T08"]
NEGATIVE = ["T02", "T04", "T06", "T07", "T09", "T10", "T11", "T12", "T14"]
kind = {codes[c]: "Praise" if c in POSITIVE else "Complaint" if c in NEGATIVE else "Other" for c in codes}
COLORS = {"Praise": "#2E8B57", "Complaint": "#C0392B", "Other": "#999999"}

# 1) How often each theme comes up, praise in green, complaints in red
shares = pd.DataFrame(theme_shares(all_tagged, 15), columns=["theme", "pct_of_reviews"])
shares["type"] = shares.theme.map(kind)
fig = px.bar(shares, x="pct_of_reviews", y="theme", orientation="h", color="type",
             color_discrete_map=COLORS,
             labels={"pct_of_reviews": "% of written reviews that mention it", "theme": "", "type": ""},
             title=f"What Michigan Domino's reviews talk about "
                   f"({all_tagged.store_id.nunique()} stores, {len(all_tagged):,} written reviews)")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=550)
fig.show()

# 2) Map: each store colored by its BIGGEST COMPLAINT, sized by % negative reviews
def biggest_complaint(df):
    c = df.theme_codes.str.split(";").explode()
    c = c[c.isin(NEGATIVE)].value_counts()
    return codes[c.index[0]] if len(c) else "None"

m = all_stores.merge(stores[["store_id", "lat", "lon"]], on="store_id")
m = m[m.n_reviews.astype(str) != "0"].copy()
m["pct_negative"] = pd.to_numeric(m.pct_negative)
m["biggest_complaint"] = [biggest_complaint(all_tagged[all_tagged.store_id == s]) for s in m.store_id]

scatter_map = getattr(px, "scatter_map", None) or px.scatter_mapbox
fig = scatter_map(m, lat="lat", lon="lon", color="biggest_complaint", size=m.pct_negative + 5,
                  hover_name="city",
                  hover_data={"store_id": True, "avg_stars": True, "pct_negative": True,
                              "lat": False, "lon": False},
                  center={"lat": 43.6, "lon": -84.8}, zoom=5.8, height=700,
                  labels={"biggest_complaint": "Biggest complaint"},
                  title="Michigan Domino's stores: biggest complaint (bigger dot = more negative reviews)")
if scatter_map is px.scatter_mapbox:
    fig.update_layout(mapbox_style="open-street-map")
fig.show()

In [ ]:
# Star rating: ALL reviews vs. written reviews vs. star-only reviews (plain Python, no Gemini)
everything = read_tab(f"full_{MY_NAME}")                     # all 9,631 reviews, including star-only
everything["stars"] = pd.to_numeric(everything.stars, errors="coerce")
everything["written"] = everything.text.astype(str).str.strip() != ""

def rating_row(label, df):
    return {"group": label, "reviews": len(df),
            "avg_stars": round(df.stars.mean(), 2),
            "pct_negative (1-2 stars)": round(100 * (df.stars <= 2).mean()),
            "pct_5_star": round(100 * (df.stars == 5).mean())}

# 1) Overall comparison
comparison = pd.DataFrame([rating_row("All reviews", everything),
                           rating_row("Written reviews", everything[everything.written]),
                           rating_row("Star-only reviews", everything[~everything.written])])
display(comparison)

# 2) Same comparison for each store
per_store = []
for sid, df in everything.groupby("store_id"):
    per_store.append({"store_id": sid,
                      "all_reviews": len(df),
                      "avg_stars_all": round(df.stars.mean(), 2),
                      "avg_stars_written": round(df[df.written].stars.mean(), 2),
                      "pct_star_only": round(100 * (~df.written).mean())})
per_store = pd.DataFrame(per_store).merge(stores[["store_id", "city"]], on="store_id")
per_store["gap"] = (per_store.avg_stars_all - per_store.avg_stars_written).round(2)

write_tab("Rating comparison", comparison)
write_tab(f"ratings_{MY_NAME}", per_store)
per_store.sort_values("gap", ascending=False)

In [ ]:
# ONE-PAGE DELIVERABLE: the most important findings on a single "Overview" tab
# Plain Python, no Gemini requests. It reads the tabs that Steps 5-7 already made.
TAB = "Overview"
W = 9   # how many columns the tab uses (A to I)

# --- Read what we already have ---
codebook = read_tab("Codebook")
codes = dict(zip(codebook.code, codebook.theme))
POSITIVE = ["T01", "T03", "T05", "T08"]
NEGATIVE = ["T02", "T04", "T06", "T07", "T09", "T10", "T11", "T12", "T14"]
kind = {codes[c]: "Praise" if c in POSITIVE else "Complaint" if c in NEGATIVE else "Other" for c in codes}

store_tab = read_tab("Stores")
county_tab = read_tab("Counties")
statewide = read_tab("Statewide")
ratings = read_tab(f"ratings_{MY_NAME}")
compare = read_tab("Rating comparison").set_index("group")
reviews = read_tab("Tagged reviews")
reviews["themes"] = reviews.themes.astype(str).replace("nan", "")


def top_theme(df, wanted):
    """Most-mentioned Praise or Complaint theme in these reviews, with its %."""
    t = df.themes.str.split(";").explode()
    t = t[t.map(kind) == wanted].value_counts()
    return f"{t.index[0]} ({round(100 * t.iloc[0] / len(df))}%)" if len(t) else ""


# % of written reviews that mention each theme
all_themes = reviews.themes.str.split(";").explode()
theme_pct = (100 * all_themes[all_themes != ""].value_counts() / len(reviews)).round().astype(int)

# One row per store that has reviews
st = ratings.merge(store_tab[["store_id", "county", "pct_negative", "summary"]], on="store_id")
st["city"] = st.city.astype(str).str.title()                 # "comstock park" -> "Comstock Park"
st["biggest_complaint"] = [top_theme(reviews[reviews.store_id == s], "Complaint") for s in st.store_id]
missing = store_tab[~store_tab.store_id.isin(st.store_id)]

# --- Build the tab row by row ---
rows, sections, headers, paragraphs = [], [], [], []

def add(*cells, style=None):
    rows.append(list(cells))
    {"section": sections, "header": headers, "paragraph": paragraphs}.get(style, []).append(len(rows))

def store_row(r):
    return (r.store_id, r.city, r.county, r.all_reviews, r.avg_stars_all, r.avg_stars_written,
            f"{r.pct_negative}%", r.biggest_complaint)

STORE_HEADER = ("Store", "City", "County", "Google reviews", "Avg stars (all)", "Avg stars (written)",
                "% negative (written)", "Biggest complaint")

add(f"Michigan Domino's Google Reviews: Overview ({len(st)} of 158 stores, preview)")
add(f"Google reviews scraped with Apify. Themes tagged by Gemini from the written reviews. "
    f"Built {pd.Timestamp.today():%B %d, %Y}.", style="paragraph")
add()

add("HEADLINE NUMBERS", style="section")
add("Measure", "Value", style="header")
n_all, n_written = int(compare.loc["All reviews", "reviews"]), int(compare.loc["Written reviews", "reviews"])
add("Stores analyzed", len(st))
add("Google reviews (all)", f"{n_all:,}")
add("Written reviews (used for themes)", f"{n_written:,} ({round(100 * n_written / n_all)}%)")
for group in ["All reviews", "Written reviews", "Star-only reviews"]:
    add(f"Average stars, {group.lower()}", compare.loc[group, "avg_stars"])
for group in ["All reviews", "Written reviews"]:
    add(f"% negative (1-2 stars), {group.lower()}", f"{compare.loc[group, 'pct_negative (1-2 stars)']}%")
add("% 5-star, all reviews", f"{compare.loc['All reviews', 'pct_5_star']}%")
add()

add("SUMMARY", style="section")
add(statewide[statewide.section == "Summary"].text.iloc[0], style="paragraph")
add()
add("KEY FINDINGS", style="section")
for text in statewide[statewide.section == "Key finding"].text:
    add("• " + text, style="paragraph")
add()
add("RECOMMENDATIONS", style="section")
for n, text in enumerate(statewide[statewide.section == "Recommendation"].text, 1):
    add(f"{n}. {text}", style="paragraph")
add()

add("WHAT CUSTOMERS TALK ABOUT", style="section")
add("Theme", "Type", "% of written reviews", style="header")
for theme, pct in theme_pct.items():
    add(theme, kind.get(theme, "Other"), f"{pct}%")
add()

big = st[st.all_reviews >= 50]                                # skip stores with very few reviews
add("TOP 5 STORES (highest average stars, 50+ reviews)", style="section")
add(*STORE_HEADER, style="header")
for r in big.nlargest(5, "avg_stars_all").itertuples():
    add(*store_row(r))
add()
add("STORES NEEDING ATTENTION (lowest average stars, 50+ reviews)", style="section")
add(*STORE_HEADER, style="header")
for r in big.nsmallest(5, "avg_stars_all").itertuples():
    add(*store_row(r))
add()

add("ALL STORES", style="section")
add(*STORE_HEADER, "Summary", style="header")
for r in st.sort_values("store_id").itertuples():
    add(*store_row(r), r.summary)
add()

add("COUNTIES (most reviews first)", style="section")
add("County", "Stores", "Written reviews", "Avg stars (written)", "% negative (written)",
    "", "", "Biggest complaint", "Summary", style="header")
county_of = reviews.drop(columns=["county"], errors="ignore").merge(stores[["store_id", "county"]], on="store_id")
for r in county_tab.itertuples():
    add(r.county, r.stores, r.n_reviews, r.avg_stars, f"{r.pct_negative}%", "", "",
        top_theme(county_of[county_of.county == r.county], "Complaint"), r.summary)
add()

add("NOTES", style="section")
notes = ["'All reviews' includes star-only reviews. Themes, % negative and summaries come from written reviews, "
         "which are more negative than star-only ones.",
         "One review can mention several themes, so theme percentages add up to more than 100%.",
         "Most counties have only 1-2 of these stores, so county results mostly reflect single stores "
         "until all 158 stores are in."]
if len(missing):
    notes.append("Not included yet: " + ", ".join(f"{r.store_id} ({r.city})" for r in missing.itertuples()) + ".")
notes.append("Details: Stores, Counties, Statewide, Tagged reviews (every review with its themes), Rating comparison.")
for text in notes:
    add("• " + text, style="paragraph")

# --- Write it to the Sheet ---
rows = [[c.item() if hasattr(c, "item") else c for c in r] + [""] * (W - len(r)) for r in rows]
try:
    ws = sheet.worksheet(TAB)
    ws.clear()
except gspread.WorksheetNotFound:
    ws = sheet.add_worksheet(title=TAB, rows=len(rows), cols=W)
ws.resize(rows=len(rows), cols=W)
ws.update(range_name="A1", values=rows, value_input_option="RAW")

# --- Make it readable: colors, bold, merged text rows, column widths ---
last = chr(ord("A") + W - 1)
BLUE = {"red": 0.0, "green": 0.39, "blue": 0.57}            # Domino's blue
GREY = {"red": 0.91, "green": 0.92, "blue": 0.93}
WHITE = {"red": 1, "green": 1, "blue": 1}
formats = [{"range": f"A1:{last}{len(rows)}",
            "format": {"textFormat": {"bold": False, "fontSize": 10, "foregroundColor": {}},
                       "backgroundColor": WHITE, "wrapStrategy": "WRAP", "verticalAlignment": "TOP"}},
           {"range": "A1", "format": {"textFormat": {"bold": True, "fontSize": 16}}}]
formats += [{"range": f"A{r}:{last}{r}", "format": {"backgroundColor": BLUE,
             "textFormat": {"bold": True, "fontSize": 11, "foregroundColor": WHITE}}} for r in sections]
formats += [{"range": f"A{r}:{last}{r}", "format": {"backgroundColor": GREY,
             "textFormat": {"bold": True}}} for r in headers]
ws.batch_format(formats)

def whole_row(r):
    return {"sheetId": ws.id, "startRowIndex": r - 1, "endRowIndex": r, "startColumnIndex": 0, "endColumnIndex": W}

widths = [230, 120, 150, 100, 100, 110, 110, 260, 520]       # pixels for columns A to I
requests = [{"unmergeCells": {"range": {"sheetId": ws.id, "startRowIndex": 0, "endRowIndex": len(rows),
                                        "startColumnIndex": 0, "endColumnIndex": W}}}]
requests += [{"mergeCells": {"range": whole_row(r), "mergeType": "MERGE_ALL"}} for r in [1] + paragraphs]
requests += [{"updateDimensionProperties": {"range": {"sheetId": ws.id, "dimension": "COLUMNS",
                                                      "startIndex": i, "endIndex": i + 1},
                                            "properties": {"pixelSize": px}, "fields": "pixelSize"}}
             for i, px in enumerate(widths)]
# merged rows don't grow to fit wrapped text, so set their height from the text length
requests += [{"autoResizeDimensions": {"dimensions": {"sheetId": ws.id, "dimension": "ROWS",
                                                      "startIndex": 0, "endIndex": len(rows)}}}]
requests += [{"updateDimensionProperties": {"range": {"sheetId": ws.id, "dimension": "ROWS",
                                                      "startIndex": r - 1, "endIndex": r},
                                            "properties": {"pixelSize": 21 * (len(str(rows[r - 1][0])) // 200 + 1)},
                                            "fields": "pixelSize"}}
             for r in paragraphs]
sheet.batch_update({"requests": requests})
ws.update_index(0)                                           # move the tab to the front
print(f"Done: '{TAB}' tab has {len(rows)} rows and is now the first tab in the Sheet.")